# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [36]:
import warnings
warnings.filterwarnings('ignore')

In [37]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN = os.getenv('HF_TOKEN')

In [38]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [39]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\r\n,I loved this product. But they only seem to l...


## LLMChain

In [40]:
%pip install langchain langchain-core langchain-openai langchain-community langchain-classic

Note: you may need to restart the kernel to use updated packages.


In [41]:
%pip show langchain langchain-core langchain-openai langchain-community

Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\santi\devtools\anaconda3\envs\ironhack\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-core
Version: 1.2.16
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: c:\Users\santi\devtools\anaconda3\envs\ironhack\Lib\site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt
---
Name: langchain-openai
Version: 1.1.10
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/provide

In [42]:
from langchain_openai import ChatOpenAI        # ✅ 1.0.3
from langchain_core.prompts import ChatPromptTemplate  # ✅ From langchain-core 1.0.5
from langchain_classic.chains import LLMChain  # ✅ Legacy (install if missing)

In [43]:
import warnings
warnings.filterwarnings("ignore")

# LangChain 1.0 imports - organized by package
from langchain_openai import OpenAI, ChatOpenAI
from langchain_community.llms import HuggingFaceHub
from langchain_community.callbacks import get_openai_callback

# Core components
from langchain_core.runnables import Runnable, RunnableSequence, RunnableParallel
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.example_selectors import LengthBasedExampleSelector

# Legacy chains (moved to langchain_classic in LangChain 1.2+)
from langchain_classic.chains import SimpleSequentialChain, LLMChain, LLMMathChain, TransformChain, SequentialChain

import inspect
import re
import os


In [44]:
#Replace None by your own value and justify
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4, # want to be true to the reviews but not too literal
    max_tokens=10
)
# llm =   OpenAI(model_name="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.4) legacy, devuelve string plano, no formato chat


In [45]:
prompt = ChatPromptTemplate.from_template(
    "Write a short description for the following product: {product}"
)

In [46]:
# chain = LLMChain(llm=llm, prompt=prompt)
chain = prompt | llm | StrOutputParser()

In [47]:
# product = #Select a product type to be described
# chain.run(product)
product = "Luxury Air Mattress"

response = chain.invoke({"product": product})
print(response)

Experience unparalleled comfort with our Luxury Air Mattress, designed


## SimpleSequentialChain

In [48]:
from langchain_classic.chains import SimpleSequentialChain  

In [49]:
llm = ChatOpenAI(
    temperature=0.9,
    max_tokens=20)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "Write a short description for the following product: {product}"
)

# Chain 1
#chain_one = LLMChain(llm=llm, prompt=first_prompt)
chain_one = first_prompt | llm | StrOutputParser()

chain_one.invoke(product)

'Indulge in ultimate comfort and relaxation with our luxury air mattress. Made with premium materials and innovative'

In [50]:
# este formato da error en esta versión
"""
second_prompt = PromptTemplate(
    input_variables=["outline"],
    template="Write a blog article in the format of the given outline" 

    Outline:
    {outline}",
)
"""

'\nsecond_prompt = PromptTemplate(\n    input_variables=["outline"],\n    template="Write a blog article in the format of the given outline" \n\n    Outline:\n    {outline}",\n)\n'

In [51]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "translate this {review} into Spanish"
)
# chain 2
# chain_two = LLMChain(llm=llm, prompt=second_prompt)
chain_two = second_prompt | llm | StrOutputParser()

In [52]:
"""
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )
                                            """
overall_simple_chain = (
    chain_one
    | (lambda review: {"review": review})
    | chain_two
)



In [53]:

"""
overall_simple_chain = chain_one | chain_two

funcionaría si solo hay uyna variable y se llama {input} a la segunda?
"""

'\noverall_simple_chain = chain_one | chain_two\n\nfuncionaría si solo hay uyna variable y se llama {input} a la segunda?\n'

In [54]:
overall_simple_chain.invoke(product)

'Disfrute del máximo confort y relajación con nuestro Colchón de Aire de Lu'

**Repeat the above twice for different products**

In [55]:
overall_simple_chain.invoke({
    "product": "Pillows insert"
    })

'Nuestros rellenos de almohada son la manera perfecta de rejuvenecer tus alm'

In [56]:
overall_simple_chain.invoke({
    "product": "Queen size sheet set"
    })

'Este lujoso juego de sábanas tamaño queen incluye una sábana ajustable'

## SequentialChain

In [57]:
from langchain_classic.chains import SequentialChain  

In [58]:
# llm = ChatOpenAI(temperature=0.9)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.4,
    max_tokens=25
)

first_prompt = ChatPromptTemplate.from_template(
  "Translate {review}"
)

"""
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key=None #Give a name to your output
                    )
                    """
chain_one = first_prompt | llm | StrOutputParser()

In [59]:
second_prompt = ChatPromptTemplate.from_template(
    "Summarize {translation}"
)

"""chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key=None #give a name to this output
                    )"""
chain_two = second_prompt | llm | StrOutputParser()

In [60]:

# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    "translate {summary} to {language}"
)
# chain 3: input= Review and output= language
"""chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key=None
                      )"""

chain_three = third_prompt | llm | StrOutputParser()

In [61]:
# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        "Share with the user the {translated_summary} in {language}"
)


"""chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key=None
                     )"""

chain_four = fourth_prompt | llm | StrOutputParser()

In [62]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
"""overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=None,
    output_variables=[None, None, None],
    verbose=True
)"""




'overall_chain = SequentialChain(\n    chains=[chain_one, chain_two, chain_three, chain_four],\n    input_variables=None,\n    output_variables=[None, None, None],\n    verbose=True\n)'

In [64]:
language = "German"

overall_chain = (

    # Step 1: review → translation
    chain_one
    | (lambda translation: {"translation": translation})

    # Step 2: translation → summary
    | chain_two
    | (lambda summary: {
        "summary": summary,
        "language": language
    })

    | chain_three
    | (lambda translated_summary: {
        "translated_summary": translated_summary,
        "language": language
    })

    # Paso 4: mensaje final
    | chain_four
)

In [65]:
review = df.Review[5]
# overall_chain(review)
overall_chain.invoke({"review":review})

'Der Geschmack ist mittelmäßig, und der Schaum hält nicht gut, was ungewöhnlich ist. Wenn ich dasselbe Produkt im'

In [66]:
review = df.Review[3]
# overall_chain(review)
overall_chain.invoke({"review":review})

'Der Artikel hebt die besten Füllungen für dekorative Kissen hervor, die auf Amazon erhältlich sind. Hier sind einige der'

In [68]:
review = df.Review[1]
# overall_chain(review)
overall_chain.invoke({"review":review})

'Die Person genoss die wasserdichte Jacke, hatte jedoch Bedenken wegen der Öffnung, da sie aus hartem'

**Repeat the above twice for different products or reviews**

## Router Chain

In [69]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [70]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [71]:
from langchain_classic.chains.router import MultiPromptChain
from langchain_classic.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain_core.prompts import PromptTemplate  # Updated path for prompts (core package)

In [72]:
llm = ChatOpenAI(temperature=0)

In [73]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [74]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [ ]:
chain.run("What is black body radiation?")

In [ ]:
chain.run("what is 2 + 2")

In [ ]:
chain.run("Why does every cell in our body contain DNA?")

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [ ]:
history_prompt = ChatPromptTemplate.from_template(
    "You are a history professor. Answer clearly:\n\n{input}"
)

history_chain = history_prompt | llm | StrOutputParser()

In [ ]:
coding_prompt = ChatPromptTemplate.from_template(
    "You are a senior software engineer. Provide a technical answer:\n\n{input}"
)

coding_chain = coding_prompt | llm | StrOutputParser()

In [ ]:
router_prompt = ChatPromptTemplate.from_template(
    """
Classify the following question into one of these categories:
- history
- coding

Only return the category name.

Question: {input}
"""
)

router_chain = router_prompt | llm | StrOutputParser()

In [ ]:
full_chain = (
    {
        "topic": router_chain,
        "input": lambda x: x["input"]
    }
    | RunnableBranch(
        (lambda x: "history" in x["topic"].lower(), history_chain),
        (lambda x: "coding" in x["topic"].lower(), coding_chain),
        history_chain  # default
    )
)

In [ ]:
print(full_chain.invoke({
    "input": "What caused World War I?"
}))

In [ ]:
print(full_chain.invoke({
    "input": "How does a binary search algorithm work?"
}))